[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/AllInVaders/aistudio-full-course/blob/main/notebooks/03_Gemini_Live_API_Tool_Agents_and_Antigravity_SDK.ipynb)

# Module 03: Gemini Live API Bidirectional Streaming, Tool Agents & Antigravity Orchestration
### Módulo 03: Gemini Live API en Tiempo Real, Agentes con Herramientas y Orquestación Antigravity

**English Overview**: Build **Stage 2 of the Flagship Project (Live Multimodal Copilot + Tool Agent)**. Connect to `gemini-3.8-live` over async bidirectional streaming, send turns with `session.send_realtime_input(...)`, invoke real-time Python tools (`calculate_unit_economics`, `check_inventory_status`), and orchestrate multi-step specialist subagents ready to graduate into **Antigravity** (`antigravity-preview-05-2026`).

**Resumen en Español**: Construye la **Etapa 2 del Proyecto Insignia (Copiloto Multimodal en Vivo + Agente con Herramientas)**. Conéctate a `gemini-3.8-live` mediante streaming bidireccional asíncrono, envía turnos con `session.send_realtime_input(...)`, ejecuta herramientas Python en tiempo real y orquesta subagentes especialistas listos para graduarse a **Antigravity** (`antigravity-preview-05-2026`).

Referencias: [Live API](https://ai.google.dev/gemini-api/docs/live-api) · [herramientas en Live API](https://ai.google.dev/gemini-api/docs/live-api/tools) · [Antigravity](https://antigravity.google)

In [ ]:
%pip install -q -U google-genai pydantic nest_asyncio

## 1. Live Model Selection & the Audio Contract / Selección de Modelo y Contrato de Audio

| Model ID | When to use it |
| :--- | :--- |
| `gemini-3.8-live` | Default real-time voice and video agent |
| `gemini-3.8-live-extended-thinking` | Background reasoning during live voice, without stalling the audio stream |
| `gemini-3.5-live-translate-preview` | Real-time speech-to-speech translation, 70+ languages |

Microphone input must be raw **16-bit PCM, 16 kHz, little-endian, mono**, streamed as
`await session.send_realtime_input(audio=types.Blob(data=chunk, mime_type='audio/pcm;rate=16000'))`.

> **Legacy migration note.** `gemini-3.8-live` is superseded by
> `gemini-3.8-live`. See https://ai.google.dev/gemini-api/docs/live-api

In [ ]:
import nest_asyncio
nest_asyncio.apply()

from google import genai
from google.genai import types

LIVE_MODEL = 'gemini-3.8-live'
LIVE_MODEL_EXTENDED_THINKING = 'gemini-3.8-live-extended-thinking'
AUDIO_MIME_TYPE = 'audio/pcm;rate=16000'   # 16-bit PCM, 16 kHz, little-endian, mono

def calculate_unit_economics(unit_cost_usd: float, retail_price_usd: float, cac_usd: float) -> dict:
    """Calculates gross margin percentage and net contribution margin per unit."""
    gross_margin = retail_price_usd - unit_cost_usd
    gross_pct = round((gross_margin / retail_price_usd) * 100.0, 2) if retail_price_usd > 0 else 0.0
    net_contribution = round(gross_margin - cac_usd, 2)
    return {
        'gross_margin_pct': gross_pct,
        'net_contribution_usd': net_contribution,
        'verdict': 'HEALTHY' if net_contribution >= 15.0 else 'TIGHT_MARGIN',
    }

def check_inventory_status(sku_code: str) -> dict:
    """Returns real-time warehouse stock status for a product SKU."""
    return {
        'sku_code': sku_code.upper(),
        'available_units': 1280,
        'warehouse': 'US-CENTRAL-FULFILLMENT',
        'lead_time_days': 4,
    }

TOOLS = {
    'calculate_unit_economics': calculate_unit_economics,
    'check_inventory_status': check_inventory_status,
}
client = genai.Client()

## 2. Live Session with Tool Calling / Sesión en Vivo con Herramientas

`send_realtime_input` is the streaming entry point. We send text here so the
notebook stays runnable without a microphone, but swapping in audio is a
one-line change (see the commented call below).

In [ ]:
async def demo_live_copilot():
    config = types.LiveConnectConfig(
        response_modalities=['TEXT'],
        tools=[calculate_unit_economics, check_inventory_status],
    )
    async with client.aio.live.connect(model=LIVE_MODEL, config=config) as session:
        prompt = 'Calculate unit economics for unit cost $28, retail price $99, and CAC $24.'

        # Text turn. For live microphone audio instead, stream PCM chunks:
        #     await session.send_realtime_input(
        #         audio=types.Blob(data=pcm_chunk, mime_type=AUDIO_MIME_TYPE)
        #     )
        await session.send_realtime_input(text=prompt)

        async for msg in session.receive():
            if msg.tool_call:
                responses = []
                for fc in msg.tool_call.function_calls:
                    out = TOOLS[fc.name](**fc.args)
                    print(f'[Tool Executed] {fc.name} -> {out}')
                    responses.append(types.FunctionResponse(id=fc.id, name=fc.name, response=out))
                await session.send_tool_response(function_responses=responses)
            if msg.server_content and msg.server_content.model_turn:
                for part in msg.server_content.model_turn.parts:
                    if part.text:
                        print(part.text, end='')
            if msg.server_content and msg.server_content.turn_complete:
                print()
                break

await demo_live_copilot()

## 3. Graduating to Antigravity / Graduación a Antigravity

The orchestration patterns above — a planner that decomposes work, specialist
subagents with isolated system instructions and tools, and a verifier gate —
are exactly what an [Antigravity](https://antigravity.google) workspace
formalizes, backed by the `antigravity-preview-05-2026` agentic coding model.

Outside the notebook, the same delegation loop runs on `gemini-3.8-flash`
through `client.interactions.create`, re-specifying `tools`,
`system_instruction`, and `generation_config` on each delegated turn. See
`labs/module-03-live-agents/antigravity_agent_demo.py` for the full harness.

In [ ]:
ORCHESTRATOR_MODEL = 'gemini-3.8-flash'
ANTIGRAVITY_MODEL = 'antigravity-preview-05-2026'

def delegate(objective: str, role_prompt: str, tools=None) -> str:
    """Runs one specialist subagent turn on the Interactions API."""
    interaction = client.interactions.create(
        model=ORCHESTRATOR_MODEL,
        input=objective,
        system_instruction=role_prompt,
        tools=tools,
        generation_config={'temperature': 0.3, 'thinking_level': 'medium'},
    )
    return interaction.output_text or ''

finding = delegate(
    objective='Stress-test pricing for AeroBrew Nano at $99 retail, $28 COGS, $24 CAC.',
    role_prompt='You are a Hardware Unit Economics Auditor. Always verify margins numerically.',
    tools=[calculate_unit_economics],
)
print(finding)
print(f'\nGraduate this harness into an Antigravity workspace ({ANTIGRAVITY_MODEL}): https://antigravity.google')